In [1]:
!pip install lpips
!pip install torchmetrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 1.9 MB/s eta 0:00:00


In [ ]:
import os
import gc
import math
import random
import csv

import numpy as np
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, Subset, WeightedRandomSampler
from torchvision.datasets import OxfordIIITPet

import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2

import lpips
from torchmetrics.image import (
    StructuralSimilarityIndexMeasure,
    PeakSignalNoiseRatio,
)


# ==========================================
# GLOBAL SETTINGS
# ==========================================
IMAGE_SIZE = 224
BATCH_SIZE = 8
NUM_EPOCHS = 20
NUM_WORKERS = 2

# For a full paper sweep, change this to:
LABEL_FRACTIONS = [0.10, 0.25, 0.50, 1.00]
# LABEL_FRACTIONS = [0.10]

# For a full paper sweep, change this to:
# SEEDS = [42, 43, 44]
SEEDS = [42]

SPLIT_SEED = 42

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

LOSS_WEIGHTS = {
    "seg": 1.0,
    "recon": 0.1,
    "distill": 0.1,
    "boundary": 1.0,
}


def seed_everything(seed=42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# ==========================================
# SPLIT / LABEL SELECTION UTILITIES
# ==========================================
def get_oxford_pet_labels(root, split, expected_len=None):
    """
    Retrieve category labels aligned with OxfordIIITPet samples.
    """
    try:
        cat_dataset = OxfordIIITPet(
            root=root,
            split=split,
            target_types="category",
            download=True,
        )

        if hasattr(cat_dataset, "_labels"):
            labels = np.asarray(cat_dataset._labels)
        else:
            labels = []
            for i in range(len(cat_dataset)):
                _, lab = cat_dataset[i]
                labels.append(lab)
            labels = np.asarray(labels)

        try:
            labels = labels.astype(np.int64)
        except Exception:
            labels = np.array(
                [abs(hash(str(x))) % 100000 for x in labels.tolist()],
                dtype=np.int64,
            )

        if expected_len is not None and len(labels) != expected_len:
            labels = np.zeros(expected_len, dtype=np.int64)

        return labels

    except Exception:
        if expected_len is None:
            expected_len = 0
        return np.zeros(expected_len, dtype=np.int64)


def stratified_train_val_indices(labels, val_fraction=0.2, seed=42):
    """
    Stratified split of trainval into train and validation.
    """
    rng = np.random.default_rng(seed)
    labels = np.asarray(labels)

    train_indices = []
    val_indices = []

    for class_id in np.unique(labels):
        class_indices = np.where(labels == class_id)[0]
        rng.shuffle(class_indices)

        if len(class_indices) <= 1:
            n_val = 0
        else:
            n_val = int(round(len(class_indices) * val_fraction))
            n_val = max(1, min(n_val, len(class_indices) - 1))

        val_indices.extend(class_indices[:n_val].tolist())
        train_indices.extend(class_indices[n_val:].tolist())

    train_indices = np.asarray(train_indices)
    val_indices = np.asarray(val_indices)

    rng.shuffle(train_indices)
    rng.shuffle(val_indices)

    return train_indices.tolist(), val_indices.tolist()


def select_label_fraction(labels, indices, fraction, seed=42):
    """
    Select a stratified subset of labeled indices from the training split.
    """
    indices = np.asarray(indices)
    labels = np.asarray(labels)

    if fraction >= 1.0:
        return indices.tolist()

    if fraction <= 0.0:
        return []

    rng = np.random.default_rng(seed)
    selected = []

    subset_labels = labels[indices]

    for class_id in np.unique(subset_labels):
        class_indices = indices[subset_labels == class_id]
        rng.shuffle(class_indices)

        n_label = int(round(len(class_indices) * fraction))
        n_label = max(1, min(n_label, len(class_indices)))

        selected.extend(class_indices[:n_label].tolist())

    selected = np.asarray(selected)
    rng.shuffle(selected)

    return selected.tolist()


# ==========================================
# DATALOADER UTILITIES
# ==========================================
def make_train_loader(
    dataset,
    indices,
    labeled_flags,
    batch_size,
    seed,
    num_workers=2,
):
    """
    Training loader for semi-supervised learning.

    If labels are scarce, labeled images are oversampled so that each batch
    is likely to contain labeled examples.
    """
    indices = np.asarray(indices)
    subset = Subset(dataset, indices.tolist())

    flags = np.asarray(labeled_flags)[indices].astype(bool)

    generator = torch.Generator().manual_seed(seed)

    def worker_init(worker_id):
        worker_seed = seed + worker_id
        random.seed(worker_seed)
        np.random.seed(worker_seed)
        torch.manual_seed(worker_seed)

    sampler = None
    shuffle = True

    num_labeled = int(flags.sum())
    num_unlabeled = int(len(flags) - num_labeled)

    if num_labeled > 0 and num_unlabeled > 0:
        label_fraction = num_labeled / float(len(flags))

        # If labels are scarce, balance batches to contain roughly 50% labeled data.
        if label_fraction < 0.5:
            unlabeled_weight = float(num_labeled) / float(max(num_unlabeled, 1))
            weights = np.where(flags, 1.0, unlabeled_weight).tolist()

            sampler = WeightedRandomSampler(
                weights=weights,
                num_samples=len(indices),
                replacement=True,
                generator=generator,
            )
            shuffle = False

    loader_kwargs = dict(
        dataset=subset,
        batch_size=batch_size,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
        worker_init_fn=worker_init,
        persistent_workers=num_workers > 0,
    )

    if sampler is not None:
        loader_kwargs["sampler"] = sampler
    else:
        loader_kwargs["shuffle"] = shuffle
        loader_kwargs["generator"] = generator

    return DataLoader(**loader_kwargs)


def make_eval_loader(
    dataset,
    indices,
    batch_size,
    seed,
    num_workers=2,
):
    indices = np.asarray(indices)
    subset = Subset(dataset, indices.tolist())

    generator = torch.Generator().manual_seed(seed)

    def worker_init(worker_id):
        worker_seed = seed + worker_id
        random.seed(worker_seed)
        np.random.seed(worker_seed)
        torch.manual_seed(worker_seed)

    return DataLoader(
        subset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
        generator=generator,
        worker_init_fn=worker_init,
        persistent_workers=num_workers > 0,
    )


# ==========================================
# TRANSFORMS
# ==========================================
def build_input_photometric():
    """
    Strong photometric/noise augmentations applied only to the model input.

    The reconstruction target does NOT receive these perturbations.
    """
    photometric = [
        A.RandomBrightnessContrast(
            brightness_limit=0.3,
            contrast_limit=0.3,
            p=0.8,
        ),
        A.HueSaturationValue(
            hue_shift_limit=20,
            sat_shift_limit=30,
            val_shift_limit=30,
            p=0.5,
        ),
        A.GaussianBlur(
            blur_limit=(3, 7),
            p=0.3,
        ),
    ]

    # Keep this optional because Albumentations versions sometimes change GaussNoise API.
    try:
        photometric.append(
            A.GaussNoise(
                var_limit=(10.0, 50.0),
                p=0.3,
            )
        )
    except Exception:
        pass

    photometric.append(A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD))
    photometric.append(ToTensorV2())

    return A.Compose(photometric)


# ==========================================
# DATASET
# ==========================================
def _binary_mask_to_long_tensor(mask):
    return (torch.as_tensor(np.asarray(mask), dtype=torch.float32) > 0.5).long()


class PetDatasetSSL(Dataset):
    """
    Semi-supervised Oxford-IIIT Pet dataset.

    Returns:
        input_image:  strongly augmented input for the segmentation model
        clean_target: clean, spatially aligned target for reconstruction / DINOv2
        mask:         foreground mask, valid only if has_label == 1
        valid:        pixels that should count in segmentation loss/metrics
        boundary:     original boundary label, valid only if has_label == 1
        label:        category label
        has_label:    1.0 if mask is labeled, else 0.0
    """

    def __init__(
        self,
        root,
        split,
        train=True,
        labels=None,
        labeled_flags=None,
    ):
        self.dataset = OxfordIIITPet(
            root=root,
            split=split,
            target_types="segmentation",
            download=True,
        )

        self.train = train

        if labels is None:
            self.labels = get_oxford_pet_labels(
                root=root,
                split=split,
                expected_len=len(self.dataset),
            )
        else:
            self.labels = np.asarray(labels, dtype=np.int64)

        assert len(self.labels) == len(self.dataset)

        if labeled_flags is None:
            self.labeled_flags = np.ones(len(self.dataset), dtype=bool)
        else:
            self.labeled_flags = np.asarray(labeled_flags, dtype=bool)

        assert len(self.labeled_flags) == len(self.dataset)

        if self.train:
            # Spatial transforms are shared by input, clean target, and masks.
            self.spatial_transform = A.Compose(
                [
                    A.Resize(IMAGE_SIZE, IMAGE_SIZE),
                    A.HorizontalFlip(p=0.5),
                    A.ShiftScaleRotate(
                        shift_limit=0.1,
                        scale_limit=0.1,
                        rotate_limit=15,
                        p=0.5,
                        border_mode=cv2.BORDER_REFLECT_101,
                    ),
                ],
                additional_targets={
                    "clean_image": "image",
                    "valid_mask": "mask",
                    "boundary_mask": "mask",
                },
            )

            self.input_photometric = build_input_photometric()

            self.clean_target_transform = A.Compose(
                [
                    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
                    ToTensorV2(),
                ]
            )

        else:
            self.val_transform = A.Compose(
                [
                    A.Resize(IMAGE_SIZE, IMAGE_SIZE),
                    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
                    ToTensorV2(),
                ],
                additional_targets={
                    "valid_mask": "mask",
                    "boundary_mask": "mask",
                },
            )

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        img, mask = self.dataset[idx]

        img = img.convert("RGB")
        img_np = np.array(img, dtype=np.uint8)
        clean_np = img_np.copy()

        mask_np = np.array(mask, dtype=np.uint8)
        if mask_np.ndim == 3:
            mask_np = mask_np[..., 0]

        label = int(self.labels[idx])
        has_label = bool(self.labeled_flags[idx])

        if self.train:
            fg_mask = (mask_np == 1).astype(np.uint8)
            valid_mask = (mask_np != 3).astype(np.uint8)
            boundary_mask = (mask_np == 3).astype(np.uint8)

            # For unlabeled images, mask supervision is disabled.
            if not has_label:
                fg_mask = np.zeros_like(mask_np, dtype=np.uint8)
                valid_mask = np.zeros_like(mask_np, dtype=np.uint8)
                boundary_mask = np.zeros_like(mask_np, dtype=np.uint8)

            spatial_aug = self.spatial_transform(
                image=img_np,
                clean_image=clean_np,
                mask=fg_mask,
                valid_mask=valid_mask,
                boundary_mask=boundary_mask,
            )

            input_img_np = spatial_aug["image"]
            clean_img_np = spatial_aug["clean_image"]

            mask_t = _binary_mask_to_long_tensor(spatial_aug["mask"])
            valid_t = _binary_mask_to_long_tensor(spatial_aug["valid_mask"])
            boundary_t = _binary_mask_to_long_tensor(spatial_aug["boundary_mask"])

            input_image = self.input_photometric(image=input_img_np)["image"]
            clean_target = self.clean_target_transform(image=clean_img_np)["image"]

            return (
                input_image,
                clean_target,
                mask_t,
                valid_t,
                boundary_t,
                label,
                torch.tensor(float(has_label), dtype=torch.float32),
            )

        else:
            fg_mask = (mask_np == 1).astype(np.uint8)
            valid_mask = (mask_np != 3).astype(np.uint8)
            boundary_mask = (mask_np == 3).astype(np.uint8)

            augmented = self.val_transform(
                image=img_np,
                mask=fg_mask,
                valid_mask=valid_mask,
                boundary_mask=boundary_mask,
            )

            input_image = augmented["image"]
            clean_target = input_image.clone()

            mask_t = _binary_mask_to_long_tensor(augmented["mask"])
            valid_t = _binary_mask_to_long_tensor(augmented["valid_mask"])
            boundary_t = _binary_mask_to_long_tensor(augmented["boundary_mask"])

            return (
                input_image,
                clean_target,
                mask_t,
                valid_t,
                boundary_t,
                label,
                torch.tensor(1.0, dtype=torch.float32),
            )


# ==========================================
# MODEL HELPERS
# ==========================================
def get_norm(num_channels):
    for groups in (32, 16, 8, 4, 1):
        if num_channels % groups == 0:
            return nn.GroupNorm(groups, num_channels)
    return nn.Identity()


class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            get_norm(out_channels),
            nn.GELU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            get_norm(out_channels),
            nn.GELU(),
        )

    def forward(self, x):
        return self.block(x)


def upsample_conv(in_channels, out_channels):
    return nn.Sequential(
        nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),
        nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
        get_norm(out_channels),
        nn.GELU(),
    )


# ==========================================
# MODEL
# ==========================================
class ConvNeXtUNet(nn.Module):
    """
    ConvNeXt-Tiny encoder + U-Net-style decoder.

    Output:
        [B, 4, H, W]
        channel 0: segmentation logits
        channels 1-3: reconstructed RGB image in [0, 1]

    Also includes a projection head for direct dense DINOv2 distillation.
    """

    def __init__(self, num_seg_classes=1, teacher_dim=384, pretrained=True):
        super().__init__()

        self.encoder = timm.create_model(
            "convnext_tiny",
            pretrained=pretrained,
            num_classes=0,
        )

        self.up3 = upsample_conv(768, 384)
        self.dec3 = DoubleConv(384 * 2, 384)

        self.up2 = upsample_conv(384, 192)
        self.dec2 = DoubleConv(192 * 2, 192)

        self.up1 = upsample_conv(192, 96)
        self.dec1 = DoubleConv(96 * 2, 96)

        self.seg_head = nn.Conv2d(96, num_seg_classes, kernel_size=1)

        self.recon_head = nn.Sequential(
            nn.Conv2d(96, 3, kernel_size=1),
            nn.Sigmoid(),
        )

        self.distill_proj = nn.Conv2d(768, teacher_dim, kernel_size=1)
        self._init_projection()

    def _init_projection(self):
        nn.init.xavier_uniform_(self.distill_proj.weight)
        if self.distill_proj.bias is not None:
            nn.init.zeros_(self.distill_proj.bias)

    def _encode(self, x):
        x = self.encoder.stem(x)
        e1 = self.encoder.stages[0](x)
        e2 = self.encoder.stages[1](e1)
        e3 = self.encoder.stages[2](e2)
        e4 = self.encoder.stages[3](e3)
        return e1, e2, e3, e4

    def _decode(self, e1, e2, e3, e4, input_size):
        d3 = self.up3(e4)
        if d3.shape[-2:] != e3.shape[-2:]:
            d3 = F.interpolate(
                d3,
                size=e3.shape[-2:],
                mode="bilinear",
                align_corners=False,
            )
        d3 = torch.cat([d3, e3], dim=1)
        d3 = self.dec3(d3)

        d2 = self.up2(d3)
        if d2.shape[-2:] != e2.shape[-2:]:
            d2 = F.interpolate(
                d2,
                size=e2.shape[-2:],
                mode="bilinear",
                align_corners=False,
            )
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2)

        d1 = self.up1(d2)
        if d1.shape[-2:] != e1.shape[-2:]:
            d1 = F.interpolate(
                d1,
                size=e1.shape[-2:],
                mode="bilinear",
                align_corners=False,
            )
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)

        d1 = F.interpolate(
            d1,
            size=input_size,
            mode="bilinear",
            align_corners=False,
        )

        return d1

    def forward(self, x, return_distill_features=False):
        input_size = x.shape[-2:]

        e1, e2, e3, e4 = self._encode(x)
        d1 = self._decode(e1, e2, e3, e4, input_size)

        seg = self.seg_head(d1)
        recon = self.recon_head(d1)

        out = torch.cat([seg, recon], dim=1)

        if return_distill_features:
            distill_features = self.distill_proj(e4)
            return out, distill_features

        return out


# ==========================================
# DINOv2 TEACHER
# ==========================================
class DINOv2Teacher(nn.Module):
    def __init__(self, device):
        super().__init__()

        self.dino = torch.hub.load(
            "facebookresearch/dinov2",
            "dinov2_vits14",
            trust_repo=True,
        ).to(device)

        self.dino.eval()
        for p in self.dino.parameters():
            p.requires_grad = False

        self.embed_dim = getattr(
            self.dino,
            "embed_dim",
            getattr(self.dino, "num_features", 384),
        )

        self.patch_size = getattr(self.dino, "patch_size", 14)
        if isinstance(self.patch_size, tuple):
            self.patch_size = self.patch_size[0]

    @torch.no_grad()
    def forward(self, x):
        features_dict = self.dino.forward_features(x)
        features = features_dict["x_norm_patchtokens"]

        B, N, C = features.shape

        H = x.shape[-2] // self.patch_size
        W = x.shape[-1] // self.patch_size

        if H * W != N:
            H = W = int(math.sqrt(N))

        features = features.transpose(1, 2).reshape(B, C, H, W)
        return features


# ==========================================
# LOSSES
# ==========================================
class MaskedDiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, pred, target, valid):
        pred = torch.sigmoid(pred)

        pred = pred.flatten(1)
        target = target.flatten(1)
        valid = valid.flatten(1)

        pred = pred * valid
        target = target * valid

        intersection = (pred * target).sum(1)
        cardinality = pred.sum(1) + target.sum(1)

        dice = (2.0 * intersection + self.smooth) / (cardinality + self.smooth)
        loss = 1.0 - dice

        has_valid = valid.sum(1) > 0
        return loss * has_valid.float()


class MaskedBCELoss(nn.Module):
    def forward(self, pred, target, valid):
        pred = pred.flatten(1)
        target = target.flatten(1)
        valid = valid.flatten(1)

        bce = F.binary_cross_entropy_with_logits(
            pred,
            target,
            reduction="none",
        )

        loss = (bce * valid).sum(1) / (valid.sum(1) + 1e-7)

        has_valid = valid.sum(1) > 0
        return loss * has_valid.float()


class SegmentationLoss(nn.Module):
    """
    Per-image segmentation loss.

    Returns:
        Tensor of shape [B]
    """

    def __init__(self, dice_weight=1.0, bce_weight=0.5):
        super().__init__()

        self.dice_loss = MaskedDiceLoss()
        self.bce_loss = MaskedBCELoss()

        self.dice_weight = dice_weight
        self.bce_weight = bce_weight

    def forward(self, pred, target, valid):
        dice = self.dice_loss(pred, target, valid)
        bce = self.bce_loss(pred, target, valid)
        return self.dice_weight * dice + self.bce_weight * bce


class BoundaryLoss(nn.Module):
    """
    Per-image boundary loss using the original boundary label.

    Returns:
        Tensor of shape [B]
    """

    def __init__(self, edge_weight=10.0):
        super().__init__()

        sobel_x = torch.tensor(
            [[[[-1, 0, 1],
               [-2, 0, 2],
               [-1, 0, 1]]]],
            dtype=torch.float32,
        )

        sobel_y = torch.tensor(
            [[[[-1, -2, -1],
               [0, 0, 0],
               [1, 2, 1]]]],
            dtype=torch.float32,
        )

        self.register_buffer("sobel_x", sobel_x)
        self.register_buffer("sobel_y", sobel_y)

        self.edge_weight = edge_weight

    def forward(self, pred_prob, boundary_target):
        boundary_target = boundary_target.float()

        # Slightly dilate boundary target into a boundary band.
        target = F.max_pool2d(
            boundary_target,
            kernel_size=3,
            stride=1,
            padding=1,
        )

        edge_x = F.conv2d(pred_prob, self.sobel_x, padding=1)
        edge_y = F.conv2d(pred_prob, self.sobel_y, padding=1)

        pred_edge = torch.sqrt(edge_x.pow(2) + edge_y.pow(2) + 1e-6)

        # Approximate normalization for 3x3 Sobel response.
        pred_edge = (pred_edge / 4.0).clamp(0.0, 1.0)

        weight = 1.0 + self.edge_weight * target
        loss_map = weight * (pred_edge - target).abs()

        loss = loss_map.flatten(1).sum(1) / (weight.flatten(1).sum(1) + 1e-7)

        has_boundary = target.flatten(1).sum(1) > 0
        return loss * has_boundary.float()


class CompositeLoss(nn.Module):
    """
    Semi-supervised composite loss.

    Labeled-only losses:
        - segmentation loss
        - boundary loss

    All-image losses:
        - denoising reconstruction loss
        - direct DINOv2 distillation loss
    """

    def __init__(
        self,
        teacher_model=None,
        lpips_loss=None,
        use_recon=False,
        use_distill=False,
        use_boundary=False,
        loss_weights=None,
    ):
        super().__init__()

        self.use_recon = use_recon
        self.use_distill = use_distill
        self.use_boundary = use_boundary

        if self.use_recon:
            assert lpips_loss is not None, "lpips_loss must be provided if use_recon=True."

        if self.use_distill:
            assert teacher_model is not None, "teacher_model must be provided if use_distill=True."

        self.teacher_model = teacher_model if self.use_distill else None
        self.lpips_loss = lpips_loss if self.use_recon else None

        self.seg_loss = SegmentationLoss()
        self.boundary_loss = BoundaryLoss()

        self.loss_weights = {
            "seg": 1.0,
            "recon": 0.1,
            "distill": 0.1,
            "boundary": 1.0,
        }

        if loss_weights is not None:
            self.loss_weights.update(loss_weights)

        self.register_buffer(
            "mean",
            torch.tensor(IMAGENET_MEAN).view(1, 3, 1, 1),
        )
        self.register_buffer(
            "std",
            torch.tensor(IMAGENET_STD).view(1, 3, 1, 1),
        )

    def forward(
        self,
        output,
        clean_target,
        target_mask,
        valid_mask,
        boundary_mask,
        has_label,
        student_distill_feat=None,
    ):
        seg_output = output[:, 0:1]
        recon_output = output[:, 1:4]

        target_mask = target_mask.unsqueeze(1).float()
        valid_mask = valid_mask.unsqueeze(1).float()
        boundary_mask = boundary_mask.unsqueeze(1).float()

        labeled = has_label.float().to(output.device)
        labeled_count = labeled.sum().clamp(min=1.0)

        zero = torch.zeros((), device=output.device, dtype=output.dtype)

        # Segmentation loss: labeled only.
        seg_per_image = self.seg_loss(seg_output, target_mask, valid_mask)
        L_seg = (seg_per_image * labeled).sum() / labeled_count

        total_loss = self.loss_weights["seg"] * L_seg

        L_recon = zero.clone()
        L_distill = zero.clone()
        L_boundary = zero.clone()

        # Boundary loss: labeled only.
        if self.use_boundary:
            seg_prob = torch.sigmoid(seg_output)
            boundary_per_image = self.boundary_loss(seg_prob, boundary_mask)
            L_boundary = (boundary_per_image * labeled).sum() / labeled_count
            total_loss = total_loss + self.loss_weights["boundary"] * L_boundary

        # Denoising reconstruction loss: all images.
        if self.use_recon:
            target_denorm = (clean_target * self.std + self.mean).clamp(0.0, 1.0)

            recon_lpips_input = recon_output * 2.0 - 1.0
            target_lpips_input = target_denorm * 2.0 - 1.0

            L_recon = self.lpips_loss(
                recon_lpips_input,
                target_lpips_input,
            ).mean()

            total_loss = total_loss + self.loss_weights["recon"] * L_recon

        # Direct DINOv2 distillation loss: all images.
        if self.use_distill:
            assert student_distill_feat is not None, (
                "student_distill_feat must be provided when use_distill=True."
            )

            with torch.no_grad():
                teacher_feat = self.teacher_model(clean_target)

            student_feat = student_distill_feat

            teacher_feat = F.interpolate(
                teacher_feat,
                size=student_feat.shape[-2:],
                mode="bilinear",
                align_corners=False,
            )

            student_feat = F.normalize(student_feat, dim=1, eps=1e-6)
            teacher_feat = F.normalize(teacher_feat, dim=1, eps=1e-6)

            cosine = (student_feat * teacher_feat).sum(dim=1)
            distill_per_image = 1.0 - cosine.mean(dim=(1, 2))

            L_distill = distill_per_image.mean()
            total_loss = total_loss + self.loss_weights["distill"] * L_distill

        return total_loss, L_seg, L_recon, L_distill, L_boundary


# ==========================================
# INFERENCE / EVALUATION
# ==========================================
def tta_predict(model, x):
    """
    Horizontal-flip test-time augmentation.
    """
    out1 = model(x)
    out2 = torch.flip(model(torch.flip(x, dims=[3])), dims=[3])

    seg1 = torch.sigmoid(out1[:, 0:1])
    seg2 = torch.sigmoid(out2[:, 0:1])
    seg = (seg1 + seg2) / 2.0

    if out1.shape[1] >= 4:
        recon = (out1[:, 1:4] + out2[:, 1:4]) / 2.0
        return torch.cat([seg, recon], dim=1)

    return seg


def evaluate_model(
    model,
    dataloader,
    device,
    use_recon=False,
    lpips_loss=None,
):
    model.eval()

    mean = torch.tensor(IMAGENET_MEAN).view(1, 3, 1, 1).to(device)
    std = torch.tensor(IMAGENET_STD).view(1, 3, 1, 1).to(device)

    ssim_metric = None
    psnr_metric = None

    if use_recon:
        ssim_metric = StructuralSimilarityIndexMeasure(data_range=1.0).to(device)
        psnr_metric = PeakSignalNoiseRatio(data_range=1.0).to(device)

        if lpips_loss is not None:
            lpips_loss.eval()

    ious = []
    dices = []
    lpips_values = []

    with torch.inference_mode():
        for data, clean_target, mask, valid, boundary, label, has_label in dataloader:
            data = data.to(device)
            clean_target = clean_target.to(device)
            mask = mask.to(device)
            valid = valid.to(device)

            output = tta_predict(model, data)

            seg_prob = output[:, 0:1]
            pred = (seg_prob.squeeze(1) > 0.5).float()

            valid_float = valid.float()

            pred_valid = pred * valid_float
            mask_valid = mask.float() * valid_float

            pred_flat = pred_valid.flatten(1)
            mask_flat = mask_valid.flatten(1)

            intersection = (pred_flat * mask_flat).sum(1)
            union = pred_flat.sum(1) + mask_flat.sum(1) - intersection

            iou = (intersection + 1e-7) / (union + 1e-7)
            dice = (2.0 * intersection + 1e-7) / (
                pred_flat.sum(1) + mask_flat.sum(1) + 1e-7
            )

            ious.append(iou.cpu())
            dices.append(dice.cpu())

            if use_recon:
                recon = output[:, 1:4]
                target_denorm = (clean_target * std + mean).clamp(0.0, 1.0)

                ssim_metric.update(recon, target_denorm)
                psnr_metric.update(recon, target_denorm)

                if lpips_loss is not None:
                    lpips_value = lpips_loss(
                        recon * 2.0 - 1.0,
                        target_denorm * 2.0 - 1.0,
                    ).mean().item()
                    lpips_values.append(lpips_value)

    results = {
        "iou": torch.cat(ious).mean().item(),
        "dice": torch.cat(dices).mean().item(),
    }

    if use_recon:
        results["ssim"] = ssim_metric.compute().item()
        results["psnr"] = psnr_metric.compute().item()
        results["lpips"] = float(np.mean(lpips_values)) if lpips_values else 0.0

    return results


# ==========================================
# TRAINING
# ==========================================
def train_model(
    model,
    train_loader,
    val_loader,
    criterion,
    device,
    num_epochs=20,
    use_recon=False,
    lpips_loss=None,
):
    encoder_params = [
        p for n, p in model.named_parameters()
        if n.startswith("encoder") and p.requires_grad
    ]

    decoder_params = [
        p for n, p in model.named_parameters()
        if not n.startswith("encoder") and p.requires_grad
    ]

    optimizer = torch.optim.AdamW(
        [
            {"params": encoder_params, "lr": 1e-5},
            {"params": decoder_params, "lr": 1e-4},
        ],
        weight_decay=1e-4,
    )

    warmup_epochs = min(3, max(0, num_epochs - 1))

    if warmup_epochs > 0:
        warmup_scheduler = torch.optim.lr_scheduler.LinearLR(
            optimizer,
            start_factor=0.01,
            total_iters=warmup_epochs,
        )

        cosine_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=num_epochs - warmup_epochs,
        )

        scheduler = torch.optim.lr_scheduler.SequentialLR(
            optimizer,
            schedulers=[warmup_scheduler, cosine_scheduler],
            milestones=[warmup_epochs],
        )
    else:
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=num_epochs,
        )

    best_iou = -float("inf")
    best_dice = 0.0

    best_state = {
        k: v.detach().cpu().clone()
        for k, v in model.state_dict().items()
    }

    for epoch in range(num_epochs):
        model.train()

        running_losses = {
            "total": 0.0,
            "seg": 0.0,
            "recon": 0.0,
            "distill": 0.0,
            "boundary": 0.0,
        }

        for data, clean_target, mask, valid, boundary, label, has_label in train_loader:
            data = data.to(device)
            clean_target = clean_target.to(device)
            mask = mask.to(device)
            valid = valid.to(device)
            boundary = boundary.to(device)
            has_label = has_label.to(device)

            optimizer.zero_grad(set_to_none=True)

            if criterion.use_distill:
                output, student_distill_feat = model(
                    data,
                    return_distill_features=True,
                )
            else:
                output = model(data)
                student_distill_feat = None

            loss, l_seg, l_recon, l_distill, l_boundary = criterion(
                output=output,
                clean_target=clean_target,
                target_mask=mask,
                valid_mask=valid,
                boundary_mask=boundary,
                has_label=has_label,
                student_distill_feat=student_distill_feat,
            )

            running_losses["total"] += loss.item()
            running_losses["seg"] += l_seg.item()
            running_losses["recon"] += l_recon.item()
            running_losses["distill"] += l_distill.item()
            running_losses["boundary"] += l_boundary.item()

            if not torch.isfinite(loss):
                continue

            if loss.requires_grad:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

        val_results = evaluate_model(
            model=model,
            dataloader=val_loader,
            device=device,
            use_recon=use_recon,
            lpips_loss=lpips_loss,
        )

        scheduler.step()

        n_batches = max(1, len(train_loader))

        msg = (
            f"Epoch {epoch + 1}/{num_epochs} | "
            f"Loss: {running_losses['total'] / n_batches:.4f} | "
            f"Seg: {running_losses['seg'] / n_batches:.4f} | "
            f"Recon: {running_losses['recon'] / n_batches:.4f} | "
            f"Distill: {running_losses['distill'] / n_batches:.4f} | "
            f"Boundary: {running_losses['boundary'] / n_batches:.4f} | "
            f"Val IoU: {val_results['iou']:.4f} | "
            f"Val Dice: {val_results['dice']:.4f}"
        )

        if use_recon:
            msg += (
                f" | SSIM: {val_results['ssim']:.4f}"
                f" | PSNR: {val_results['psnr']:.2f}"
                f" | LPIPS: {val_results['lpips']:.4f}"
            )

        print(msg, flush=True)

        if val_results["iou"] > best_iou:
            best_iou = val_results["iou"]
            best_dice = val_results["dice"]

            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }

    return best_state, best_iou, best_dice


# ==========================================
# CSV LOGGER
# ==========================================
def append_results_csv(csv_path, row):
    file_exists = os.path.exists(csv_path)

    with open(csv_path, "a", newline="") as f:
        writer = csv.writer(f)

        if not file_exists:
            writer.writerow(
                [
                    "label_fraction",
                    "seed",
                    "config",
                    "best_iou",
                    "best_dice",
                ]
            )

        writer.writerow(row)


# ==========================================
# MAIN
# ==========================================
if __name__ == "__main__":
    seed_everything(SPLIT_SEED)

    os.makedirs("./data", exist_ok=True)
    os.makedirs("./checkpoints", exist_ok=True)
    os.makedirs("./outputs", exist_ok=True)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}", flush=True)

    print("Creating fixed stratified train/validation split...", flush=True)

    base_dataset = OxfordIIITPet(
        root="./data",
        split="trainval",
        target_types="segmentation",
        download=True,
    )

    total_size = len(base_dataset)
    labels = get_oxford_pet_labels(
        root="./data",
        split="trainval",
        expected_len=total_size,
    )

    train_idx, val_idx = stratified_train_val_indices(
        labels=labels,
        val_fraction=0.2,
        seed=SPLIT_SEED,
    )

    print(f"Total trainval images: {total_size}", flush=True)
    print(f"Training images: {len(train_idx)}", flush=True)
    print(f"Validation images: {len(val_idx)}", flush=True)

    val_dataset = PetDatasetSSL(
        root="./data",
        split="trainval",
        train=False,
        labels=labels,
        labeled_flags=np.ones(total_size, dtype=bool),
    )

    configs = [
        {
            "name": "SegOnly",
            "recon": False,
            "distill": False,
            "boundary": False,
        },
        {
            "name": "PlusRecon",
            "recon": True,
            "distill": False,
            "boundary": False,
        },
        {
            "name": "PlusBoundary",
            "recon": False,
            "distill": False,
            "boundary": True,
        },
        {
            "name": "PlusDistill",
            "recon": False,
            "distill": True,
            "boundary": False,
        },
        {
            "name": "Full",
            "recon": True,
            "distill": True,
            "boundary": True,
        },
    ]

    any_distill = any(cfg["distill"] for cfg in configs)
    any_recon = any(cfg["recon"] for cfg in configs)

    teacher_model = None
    teacher_dim = 384

    if any_distill:
        print("Loading DINOv2 teacher...", flush=True)
        teacher_model = DINOv2Teacher(device).to(device)
        teacher_dim = teacher_model.embed_dim

    perceptual_loss = None

    if any_recon:
        print("Loading LPIPS perceptual loss...", flush=True)
        perceptual_loss = lpips.LPIPS(net="vgg").to(device)
        perceptual_loss.eval()

        for p in perceptual_loss.parameters():
            p.requires_grad = False

    results_csv = "./outputs/results.csv"
    summary = []

    for training_seed in SEEDS:
        print("\n" + "=" * 70)
        print(f"SEED: {training_seed}")
        print("=" * 70, flush=True)

        seed_everything(training_seed)

        # Create reference initialization for this seed.
        reference_model = ConvNeXtUNet(
            num_seg_classes=1,
            teacher_dim=teacher_dim,
            pretrained=True,
        ).to(device)

        reference_state = {
            k: v.detach().cpu().clone()
            for k, v in reference_model.state_dict().items()
        }

        del reference_model
        torch.cuda.empty_cache()
        gc.collect()

        for label_fraction in LABEL_FRACTIONS:
            label_tag = f"{int(round(label_fraction * 100))}pct"

            print("\n" + "=" * 70)
            print(f"LABEL FRACTION: {label_fraction:.2f}")
            print("=" * 70, flush=True)

            labeled_train_idx = select_label_fraction(
                labels=labels,
                indices=train_idx,
                fraction=label_fraction,
                seed=SPLIT_SEED,
            )

            labeled_flags = np.zeros(total_size, dtype=bool)
            labeled_flags[labeled_train_idx] = True

            num_train = len(train_idx)
            num_labeled = int(labeled_flags[np.asarray(train_idx)].sum())

            print(f"Training split size: {num_train}", flush=True)
            print(f"Labeled training images: {num_labeled}", flush=True)
            print(f"Unlabeled training images: {num_train - num_labeled}", flush=True)

            train_dataset = PetDatasetSSL(
                root="./data",
                split="trainval",
                train=True,
                labels=labels,
                labeled_flags=labeled_flags,
            )

            for cfg in configs:
                print("\n" + "-" * 70)
                print(f"Training: {cfg['name']} | Seed: {training_seed} | Labels: {label_tag}")
                print("-" * 70, flush=True)

                # Reset RNG for each ablation so configs are compared fairly.
                seed_everything(training_seed)

                train_loader = make_train_loader(
                    dataset=train_dataset,
                    indices=train_idx,
                    labeled_flags=labeled_flags,
                    batch_size=BATCH_SIZE,
                    seed=training_seed,
                    num_workers=NUM_WORKERS,
                )

                val_loader = make_eval_loader(
                    dataset=val_dataset,
                    indices=val_idx,
                    batch_size=BATCH_SIZE,
                    seed=training_seed,
                    num_workers=NUM_WORKERS,
                )

                model = ConvNeXtUNet(
                    num_seg_classes=1,
                    teacher_dim=teacher_dim,
                    pretrained=False,
                ).to(device)

                model.load_state_dict(reference_state)

                criterion = CompositeLoss(
                    teacher_model=teacher_model if cfg["distill"] else None,
                    lpips_loss=perceptual_loss if cfg["recon"] else None,
                    use_recon=cfg["recon"],
                    use_distill=cfg["distill"],
                    use_boundary=cfg["boundary"],
                    loss_weights=LOSS_WEIGHTS,
                ).to(device)

                best_state, best_iou, best_dice = train_model(
                    model=model,
                    train_loader=train_loader,
                    val_loader=val_loader,
                    criterion=criterion,
                    device=device,
                    num_epochs=NUM_EPOCHS,
                    use_recon=cfg["recon"],
                    lpips_loss=perceptual_loss if cfg["recon"] else None,
                )

                ckpt_path = (
                    f"checkpoints/"
                    f"label_{label_tag}_"
                    f"seed_{training_seed}_"
                    f"{cfg['name']}.pth"
                )

                torch.save(best_state, ckpt_path)

                print(f"Saved checkpoint: {ckpt_path}", flush=True)
                print(f"Best IoU: {best_iou:.4f}", flush=True)
                print(f"Best Dice: {best_dice:.4f}", flush=True)

                append_results_csv(
                    results_csv,
                    [
                        label_fraction,
                        training_seed,
                        cfg["name"],
                        best_iou,
                        best_dice,
                    ],
                )

                summary.append(
                    f"Seed {training_seed} | Labels {label_tag} | "
                    f"{cfg['name']} | IoU {best_iou:.4f} | Dice {best_dice:.4f}"
                )

                del model, criterion, train_loader, val_loader, best_state
                torch.cuda.empty_cache()
                gc.collect()

            del train_dataset
            torch.cuda.empty_cache()
            gc.collect()

    print("\n" + "=" * 70)
    print("FINAL SUMMARY")
    print("=" * 70, flush=True)

    for line in summary:
        print(line, flush=True)

    print("\nSCRIPT COMPLETED SUCCESSFULLY", flush=True)

Using device: cuda
Creating fixed stratified train/validation split...


100%|██████████| 792M/792M [00:04<00:00, 191MB/s]  
100%|██████████| 19.2M/19.2M [00:00<00:00, 106MB/s] 


Total trainval images: 3680
Training images: 2944
Validation images: 736
Loading DINOv2 teacher...
Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip


/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vits14/dinov2_vits14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vits14_pretrain.pth


...

Loading model from: /usr/local/lib/python3.12/dist-packages/lpips/weights/v0.1/vgg.pth

SEED: 42


model.safetensors:   0%|          | 0.00/114M [00:00<?, ?B/s]


LABEL FRACTION: 0.10
Training split size: 2944
Labeled training images: 295
Unlabeled training images: 2649

----------------------------------------------------------------------
Training: SegOnly | Seed: 42 | Labels: 10pct
----------------------------------------------------------------------


/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
/tmp/ipykernel_58/2343989788.py:305: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(


Epoch 1/20 | Loss: 0.7063 | Seg: 0.7063 | Recon: 0.0000 | Distill: 0.0000 | Boundary: 0.0000 | Val IoU: 0.7462 | Val Dice: 0.8404
Epoch 2/20 | Loss: 0.2248 | Seg: 0.2248 | Recon: 0.0000 | Distill: 0.0000 | Boundary: 0.0000 | Val IoU: 0.9516 | Val Dice: 0.9723
Epoch 3/20 | Loss: 0.1468 | Seg: 0.1468 | Recon: 0.0000 | Distill: 0.0000 | Boundary: 0.0000 | Val IoU: 0.9640 | Val Dice: 0.9791
Epoch 4/20 | Loss: 0.1072 | Seg: 0.1072 | Recon: 0.0000 | Distill: 0.0000 | Boundary: 0.0000 | Val IoU: 0.9638 | Val Dice: 0.9789
Epoch 5/20 | Loss: 0.0866 | Seg: 0.0866 | Recon: 0.0000 | Distill: 0.0000 | Boundary: 0.0000 | Val IoU: 0.9682 | Val Dice: 0.9811
Epoch 6/20 | Loss: 0.0687 | Seg: 0.0687 | Recon: 0.0000 | Distill: 0.0000 | Boundary: 0.0000 | Val IoU: 0.9741 | Val Dice: 0.9848
Epoch 7/20 | Loss: 0.0535 | Seg: 0.0535 | Recon: 0.0000 | Distill: 0.0000 | Boundary: 0.0000 | Val IoU: 0.9730 | Val Dice: 0.9841
Epoch 8/20 | Loss: 0.0506 | Seg: 0.0506 | Recon: 0.0000 | Distill: 0.0000 | Boundary: 0.00